# CARDIOVERSE: Reaction Network Explanations
This sandbox demonstrates the complete explainability pipeline for identifying informative metabolic reactions in cardiotoxicity prediction.

**Goal:** Load a trained model and its validation set, generate Integrated Gradients (IG) attributions, and perform differential analysis to identify significantly relevant reactions.

In [17]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import functional as F
import torch_geometric as tg
from tqdm import tqdm
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from cardioverse.models.linet import LiNetModel
from cardioverse.configs.gnn_config import GNNModelConfig
from cardioverse.explanations.integrated_gradients import IGExplainer

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Load Trained Model and Metadata
We use the bundled artifact that contains the architecture configuration, trained weights, and the specific validation sample IDs used during training.

In [18]:
def load_linet_bundle(artefact_path):
    """Reconstructs the model and metadata from the bundled artifact."""
    bundle = torch.load(artefact_path, map_location="cpu", weights_only=False)
    
    # Initialize model
    config = GNNModelConfig(**bundle["model_config"])
    model = LiNetModel(config)
    model.load_state_dict(bundle["model_state_dict"])
    model.eval()
    
    return model, config, bundle.get("val_sample_ids", [])

# Point to the latest trained artifact
ARTEFACT_PATH = "../artefacts/reactions_linet_20260330_100639/trained_model.pt"

model, model_config, val_sample_ids = load_linet_bundle(ARTEFACT_PATH)
model = model.to(DEVICE)
print(f"Model and {len(val_sample_ids)} validation IDs loaded from {ARTEFACT_PATH}")

Model and 53 validation IDs loaded from ../artefacts/reactions_linet_20260330_100639/trained_model.pt


## Load and Prepare Validation Data
We load the metabolic reaction features and the graph structure, then filter to exactly match the validation set.

In [21]:
# Load data files
X_df = pd.read_csv("../data/reaction_level_data/X.csv", index_col=0)
edge_index = np.load("../data/reaction_level_data/edge_index.npy")
reactions = list(X_df.columns)

# Load labels
drug_metadata = pd.read_csv("../data/metadata/drug_metadata.csv")
drug_tox_map = drug_metadata.set_index("Drug name")["Is_cardiotoxic"].to_dict()

# Filter to the exact validation samples saved in the bundle
X_val_df = X_df.loc[val_sample_ids]
y_val = np.array([1 if drug_tox_map[name.split(":")[-1]] == "Yes" else 0 for name in val_sample_ids])

# Create Dataset
val_dataset = TensorDataset(
    torch.from_numpy(X_val_df.values).float(),
    torch.from_numpy(y_val).long(),
    torch.arange(len(X_val_df))
)

print(f"Validation set prepared: {len(val_dataset)} samples.")

Validation set prepared: 53 samples.


## Identify Correct Predictions
Differential attribution analysis is most informative when comparing correctly classified samples.

In [24]:
yval_true, yval_pred = [], []
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
ei_torch = torch.from_numpy(edge_index).long().to(DEVICE)

with torch.no_grad():
    for batch in val_loader:
        x, y, _ = batch
        # Construct PyG batch manually for prediction
        data_list = [tg.data.Data(x=x[i].unsqueeze(1), edge_index=ei_torch) for i in range(len(x))]
        graph_batch = tg.data.Batch.from_data_list(data_list).to(DEVICE)
        logits, _ = model(graph_batch.x, graph_batch.edge_index, graph_batch.batch)
        
        yval_pred += list(F.softmax(logits, dim=1).argmax(1).cpu().numpy())
        yval_true += list(y.numpy())

val_results_df = pd.DataFrame(index=val_sample_ids, data={"ytrue": yval_true, "ypred": yval_pred})
val_results_df.head()

,ytrue,ypred
MSN01:afatinib,0,0
MSN02:afatinib,0,0
MSN05:afatinib,0,1
MSN06:afatinib,0,0
MSN08:afatinib,0,0


## Generate Attributions (Integrated Gradients)
We compute IG scores for all validation samples.

In [26]:
explainer = IGExplainer(model, edge_index=torch.from_numpy(edge_index).long(), device=DEVICE)

print("Generating attributions for the full validation set...")
ig_df = explainer.explain(
    val_dataset, 
    target=1,               # Attribute towards 'Toxic' class
    feature_names=reactions,
    internal_batch_size=None,
)
ig_df.index = val_sample_ids
print("Attributions generated.")

Generating attributions for the full validation set...


100%|███████████████████████████████████████████████████████████████████████| 53/53 [00:07<00:00,  7.20it/s]

Attributions generated.


## Differential Attribution Analysis
We perform Mann-Whitney U tests to find reactions with significantly different attributions between Toxic and Non-Toxic samples.

In [27]:
# Filter to correct predictions
correct_tox_ids = val_results_df[(val_results_df.ytrue == 1) & (val_results_df.ytrue == val_results_df.ypred)].index
correct_notox_ids = val_results_df[(val_results_df.ytrue == 0) & (val_results_df.ytrue == val_results_df.ypred)].index

ig_tox = ig_df.loc[correct_tox_ids]
ig_notox = ig_df.loc[correct_notox_ids]

print(f"Analyzing {len(ig_tox)} True Positives vs {len(ig_notox)} True Negatives...")

pvals = []
for reaction in tqdm(ig_df.columns):
    _, p = mannwhitneyu(ig_tox[reaction], ig_notox[reaction], alternative='two-sided')
    pvals.append(p)

# Correct for multiple hypothesis testing (FDR BH)
reject, pvals_adj, _, _ = multipletests(pvals, method='fdr_bh')

# Compile full results
infr_df = pd.DataFrame({
    'pval': pvals,
    'pval_adj': pvals_adj,
    '-log10padj': -np.log10(pvals_adj),
    'reject': reject,
    'mean_Tox': ig_tox.mean(),
    'mean_NoTox': ig_notox.mean(),
    'mean_diff': ig_tox.mean() - ig_notox.mean()
}, index=reactions).sort_values('pval_adj')

Analyzing 20 True Positives vs 24 True Negatives...


100%|█████████████████████████████████████████████████████████████████| 3572/3572 [00:02<00:00, 1660.99it/s]


## Top 20 Significant Reactions
Identify the top 20 reactions most significantly associated with the model's toxicity predictions.

In [28]:
reactions_meta = pd.read_csv("../data/metadata/reactions_metadata.csv").set_index("Abbreviation")
top20 = infr_df.head(20).join(reactions_meta[["Description", "Subsystem"]], how="left")

pd.options.display.float_format = "{:.2e}".format
print("Top 20 Most Informative Reactions:")
top20

Top 20 Most Informative Reactions:


,pval,pval_adj,-log10padj,reject,mean_Tox,mean_NoTox,mean_diff,Description,Subsystem
RCR11266,8.22e-08,9.52e-05,4.02e+00,True,4.30e-03,4.37e-07,4.30e-03,L-ornithine carboxy-lyase (putrescine-forming),Arginine and proline metabolism
RCR12498,6.32e-08,9.52e-05,4.02e+00,True,-7.79e-03,-1.29e-05,-7.77e-03,butyrinase,Fatty acid metabolism
RCR13159,1.07e-07,9.52e-05,4.02e+00,True,-1.56e-03,-1.91e-06,-1.56e-03,CDP-diacylglycerol:phosphatidylglycerol 3-phos...,Glycerolipid metabolism
RCR10111,6.32e-08,9.52e-05,4.02e+00,True,8.69e-04,-3.84e-07,8.70e-04,ATP diphosphohydrolase (diphosphate-forming),Nucleotide metabolism
RCR20268,1.78e-07,1.27e-04,3.90e+00,True,1.09e-03,3.68e-05,1.06e-03,(R)-3-hydroxybutanoate / H+ symport [c=m],Transport
RCR14452,2.95e-07,1.76e-04,3.76e+00,True,3.57e-04,3.58e-06,3.53e-04,"alpha-1,6-mannosyl-glycoprotein 6-beta-N-acety...",Glycan metabolism
RCR12902,1.61e-06,2.32e-04,3.64e+00,True,-8.47e-04,2.38e-07,-8.47e-04,ATP:(R)-mevalonate 5-phosphotransferase,Cholesterol metabolism
RCR14256,1.27e-06,2.32e-04,3.64e+00,True,1.24e-03,1.48e-06,1.24e-03,ATP:deoxycitidine 5'-phosphotransferase[c],Pyrimidine metabolism
RCR10046,6.99e-07,2.32e-04,3.64e+00,True,2.11e-03,-1.54e-05,2.13e-03,oleoyl-CoA synthetase,Fatty acid metabolism
RCR13381,1.43e-06,2.32e-04,3.64e+00,True,1.01e-03,-1.99e-06,1.01e-03,Heparosan-N-sulfate D-glucuronate glucurono-5-...,Proteoglycan metabolism
